In [1]:
import tensorrt as trt
import os.path

In [ ]:
logger = trt.Logger(trt.Logger.WARNING)
explicit_batch = 1 << (int)(trt.NetworkDefinitionCreationFlag.EXPLICIT_BATCH)  # trt7
batch_size = 4
model = "../Models/beachbot_yolov5s_beach-cleaning-object-detection__v8-yolotrain__yolov5pytorch_1280_finetune/best.onnx"
output = model.rsplit(".",1)[0] + ".engine"


if os.path.isfile(output):
    print(f"Output file {output} exists, skipping model conversion!")
else:
    print("Create tensorrt file", output)

    with trt.Builder(logger) as builder:
        with builder.create_network(explicit_batch) as network:
            with trt.OnnxParser(network, logger) as parser:
                with builder.create_builder_config() as builder_config:
                    #builder.fp16_mode = True # optional
                    #builder_config.max_workspace_size = workspace_size * (1024 * 1024)
                    builder_config.set_flag(trt.BuilderFlag.FP16)
                    #builder_config.set_flag(trt.BuilderFlag.INT8)
                    with open(model, 'rb') as f:
                        print('Beginning ONNX file parsing')
                        if not parser.parse(f.read()):
                            for error in range(parser.num_errors):
                                print("ERROR", parser.get_error(error))
                    print("num layers:", network.num_layers)
                    print("out shape:", network.get_input(0).shape)
                    #network.get_input(0).shape = [batch_size, 3, 608, 608]  # trt7
                    engine = builder.build_serialized_network(network, builder_config)
                    if engine is not None:
                        #engine = builder.build_cuda_engine(network)
                        with open(output, 'wb') as f:
                            f.write(engine)
                        print("Completed creating Engine")
                    else:
                        print(f"Error, model conversion failed, engin is none.")




In [7]:
import cv2
#import pycuda.autoinit
import pycuda.driver as cuda
import numpy as np
import time

img = cv2.imread('../Datasets/beach-cleaning-object-detection.v8-yolotrain.yolov5pytorch/test/images/21_jpg.rf.6514a882ca7274f36b960610d8cc6950.jpg')
print(img.shape)
#img = cv2.resize(img, (800, 1280))

print(img.shape, len(img))
TRT_LOGGER = trt.Logger(trt.Logger.INFO)
TRTbin = '../Models/beachbot_yolov5s_beach-cleaning-object-detection__v8-yolotrain__yolov5pytorch_1280_finetune/best.engine'
print('trtbin', TRTbin)
with open(TRTbin, 'rb') as f, trt.Runtime(TRT_LOGGER) as runtime:
    engine = runtime.deserialize_cuda_engine(f.read())
context = engine.create_execution_context()
inputs, outputs, bindings = [], [], []

cuda.init() 
device = cuda.Device(0) 
ctx = device.make_context() 
stream = cuda.Stream()

for binding in engine:
    # size = trt.volume(engine.get_binding_shape(binding))
    # dtype = trt.nptype(engine.get_binding_dtype(binding))
    size = trt.volume(engine.get_tensor_shape(binding))
    print("binding size", size)
    dtype = trt.nptype(engine.get_tensor_dtype(binding))
    host_mem = cuda.pagelocked_empty(size, dtype)
    device_mem = cuda.mem_alloc(host_mem.nbytes)
    bindings.append(int(device_mem))
    if engine.get_tensor_mode(binding) == trt.TensorIOMode.INPUT:
    #if engine.binding_is_input(binding):
        inputs.append({ 'host': host_mem, 'device': device_mem })
    else:
        outputs.append({ 'host': host_mem, 'device': device_mem })
# # save to class
# inputs = inputs
# outputs = outputs
# bindings = bindings
# stream = stream
# post processing config
filters = (80 + 5) * 3
output_shapes = [
    (1, 3, 80, 80, 85),
    (1, 3, 40, 40, 85),
    (1, 3, 20, 20, 85)
]
strides = np.array([8., 16., 32.])
anchors = np.array([
    [[10,13], [16,30], [33,23]],
    [[30,61], [62,45], [59,119]],
    [[116,90], [156,198], [373,326]],
])
nl = len(anchors)
nc = 6 # classes
no = nc + 5 # outputs per anchor
na = len(anchors[0])
a = anchors.copy().astype(np.float32)
a = a.reshape(nl, -1, 2)
anchors = a.copy()
anchor_grid = a.copy().reshape(nl, 1, -1, 1, 1, 2)


def format_yolov5(frame):
    row, col, _ = frame.shape
    _max = max(col, row)
    result = np.zeros((_max, _max, 3), np.uint8)
    result[0:row, 0:col] = frame
    return result


shape_orig_WH = (img.shape[1], img.shape[0])

#img=format_yolov5(img)

print('original image shape', img.shape)
#img = cv2.resize(img, (800, 1280))
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
# img = img.transpose((2, 0, 1)).astype(np.float16)
img = img.transpose((2, 0, 1)).astype(np.float16)
img /= 255.0




# copy img to input memory
# self.inputs[0]['host'] = np.ascontiguousarray(img)
inputs[0]['host'] = np.ravel(img)
# transfer data to the gpu
for inp in inputs:
    cuda.memcpy_htod_async(inp['device'], inp['host'], stream)
# run inference
stream.synchronize()
start = time.time()
ctx.pop() 
context.execute_v2(bindings=bindings) # stream_handle=stream.handle, 

#context.execute_async_v3(stream_handle=stream.handle) # bindings=bindings,
# fetch outputs from gpu
for out in outputs:
    cuda.memcpy_dtoh_async(out['host'], out['device'], stream)
# synchronize stream
stream.synchronize()
end = time.time()
print('execution time:', end-start)
outputs = [out['host'] for out in outputs]



# reshape from flat to (1, 3, x, y, 85)
reshaped = []
for output, shape in zip(outputs, output_shapes):
    print(output.shape)
    rescaled = outputs[-1].reshape((1, -1, nc + 5))
    print(rescaled.shape)
    #reshaped.append(output.reshape(shape))


print(reshaped)
ctx.pop() 



(800, 1280, 3)
(800, 1280, 3) 800
trtbin ../Models/beachbot_yolov5s_beach-cleaning-object-detection__v8-yolotrain__yolov5pytorch_1280_finetune/best.engine
[03/10/2025-11:24:01] [TRT] [I] The logger passed into createInferRuntime differs from one already provided for an existing builder, runtime, or refitter. Uses of the global logger, returned by nvinfer1::getLogger(), will return the existing value.
[03/10/2025-11:24:01] [TRT] [I] Loaded engine size: 17 MiB
[03/10/2025-11:24:01] [TRT] [I] [MemUsageChange] TensorRT-managed allocation in IExecutionContext creation: CPU +0, GPU +41, now: CPU 0, GPU 109 (MiB)
binding size 3072000
binding size 693000
original image shape (800, 1280, 3)
execution time: 0.4795539379119873
(693000,)
(1, 63000, 11)
[]


In [11]:
rescaled

array([[[5.4609e+00, 6.6172e+00, 1.4195e+01, ..., 1.6403e-02,
         5.5273e-01, 6.3721e-02],
        [1.1609e+01, 7.0234e+00, 2.6781e+01, ..., 2.2125e-02,
         3.7061e-01, 3.7750e-02],
        [1.8578e+01, 6.9141e+00, 3.4531e+01, ..., 1.4900e-02,
         3.5229e-01, 2.9251e-02],
        ...,
        [1.1920e+03, 7.6150e+02, 1.9738e+02, ..., 9.9976e-02,
         8.2910e-01, 2.9251e-02],
        [1.2190e+03, 7.6050e+02, 1.3188e+02, ..., 1.3293e-01,
         8.0029e-01, 4.1321e-02],
        [1.2520e+03, 7.6300e+02, 1.4688e+02, ..., 1.9666e-01,
         5.6201e-01, 8.2397e-02]]], dtype=float16)

In [3]:
import cv2

ModuleNotFoundError: No module named 'cv2'